# Step 2: Process Features
Ce notebook évalue sur chaque segment du réseaux chaque attribut. La méthode de comptabilisation de chaque attribut est prédéfinie dans le fichier excel attributs_info.

## Traitement des attributs

In [1]:
%load_ext autoreload
%autoreload 2


import geopandas as gpd
import pandas as pd
from functools import reduce
import osmnx as ox
from shapely.geometry import Point
from matplotlib import pyplot as plt

import os, time
import numpy as np
import geopandas as gpd
import shapely


import os

# from method_a_buffer import extract_buffer_feature
from feature_to_network import extract_buffer_feature
from feature_query import make_feature_query

import geopandas as gpd
import folium
from folium import Choropleth, CircleMarker, GeoJson
import osmnx as ox
import branca.colormap as cm
import rasterio
# Display the map in the notebook
from IPython.display import display
# Display all columns in a dataframe
pd.set_option('display.max_columns', None)


operation_crs = "EPSG:2056"  # Swiss coordinate system
target_crs = "EPSG:4326"  # WGS84 coordinate system

print("Set parameters : GG or GE, bike or walk")
territory = 'GG' # GG or GE
network = "walk"  # walk or bike


input_file_path = '../../Data/input'
output_step1_path=f"../../Data/output/{network}/{territory}/step-1"
output_step2_path=f"../../Data/output/{network}/{territory}/step-2"
output_step3_path=f"../../Data/output/{network}/{territory}/step-3"
save_path = f"../../Data/output/{network}/{territory}/step-2"
attribute_source_path = f"source_path_{territory}"
file_name = f"file_name_{territory}"  # column name in attributs_info excel
how_name = f"how_{territory}"
geometry_type_name = f"geometry_type_{territory}"
value_column_name = f"value_column_{territory}"
include = f"include_in_index_{territory}"

# Load segments GeoDataFrame (with 'segment_id')
print("Loading pedestrian segments...")
# reLoad pedestrian segments
segmented_net = gpd.read_parquet(os.path.join(output_step1_path, "step1_all_segments.parquet"))
segmented_net["geometry"] = segmented_net["geometry"].apply(lambda geom: geom.buffer(0) if not geom.is_valid else geom)
segmented_net = segmented_net.to_crs(operation_crs)


Set parameters : GG or GE, bike or walk
Loading pedestrian segments...


In [2]:
attributs_info = pd.read_excel(f"{input_file_path}/attributs/attributs_info_{network}_unweighted.xlsx", sheet_name="attributs_info")
attributs_info = attributs_info.loc[~attributs_info[include].isin(['False', 'Unavailable'])].copy()  # Filter out rows where include_in_index is False or 'Unavailable'

attributs_info

,Class,meta_attribute,attribute,include_in_index_GG,include_in_index_GE,source_type_GE,source_path_GE,file_name_GE,source_type_VD,source_path_VD,file_name_VD,source_type_FR,source_path_FR,file_name_FR,source_type_GG,source_path_GG,file_name_GG,merge_rule_GG,initial_weight,class_weight,buffer_size_GG,buffer_size_GE,impact_attribut,geometry_type_GE,how_GE,value_column_GE,geometry_type_GG,how_GG,value_column_GG,clip_GG,clip_GE,missing_value_strategy,filter_column,filter_values,crs,save_format,Commentaire
0,Attractivité,lac_cours_deau,lac_cours_deau,True,True,sitg,attributs/GE/lac_cours_deau,CAD_NATURE_SOL-SHP/CAD_NATURE_SOL.shp,NaN,NaN,NaN,NaN,NaN,NaN,osm,attributs/GG/osm,osm_attributes.gpkg,NaN,0.5,0.5,50,50,favorable,polygon,presence,NaN,polygon,presence,NaN,NaN,NaN,zero_if_missing,filtered,1,2056,"parquet, csv, gpkg",ok
1,Attractivité,fontaine,fontaine,True,True,osm,attributs/GG/osm,osm_attributes.gpkg,NaN,NaN,NaN,NaN,NaN,NaN,osm,attributs/GG/osm,osm_attributes.gpkg,NaN,0.5,0.5,50,50,favorable,point,count,NaN,point,count,NaN,2.0,2.0,zero_if_missing,filtered,1,2056,"parquet, csv, gpkg",ok
2,Attractivité,espaces_ouverts,espaces_ouverts,True,True,sitg,attributs/GE/espaces_ouverts,OBS_EQUIPEMENTS_ESPACES_PUB-SHP/OBS_EQUIPEMENT...,NaN,NaN,NaN,NaN,NaN,NaN,osm,attributs/GG/osm,osm_attributes.gpkg,NaN,0.5,0.5,1,1,favorable,polygon,presence,NaN,polygon,presence,NaN,NaN,2.0,zero_if_missing,filtered,1,2056,"parquet, csv, gpkg",TB
3,Attractivité,amenite,rez_actif,True,True,sitg,attributs/GE/rez_actif,REG_ENTREPRISE_ETABLISSEMENT-SHP/REG_ENTREPRIS...,NaN,NaN,NaN,NaN,NaN,NaN,osm,attributs/GG/osm,osm_attributes.gpkg,NaN,0.5,0.5,50,50,favorable,point,count,NaN,point,count,NaN,4.0,10.0,zero_if_missing,filtered,1,2056,"parquet, csv, gpkg",TB
4,Attractivité,transport_public,tp,True,True,sitg,attributs/GE/tp,TPG_ARRETS-SHP/TPG_ARRETS.shp,NaN,NaN,NaN,NaN,NaN,NaN,osm,attributs/GG/osm,osm_attributes.gpkg,NaN,0.5,0.5,150,150,favorable,point,presence,NaN,point,presence,NaN,NaN,NaN,zero_if_missing,filtered,1,2056,"parquet, csv, gpkg",TB
5,Attractivité,amenite,amenite,True,True,sitg,attributs/GE/amenite,REG_ENTREPRISE_ETABLISSEMENT-SHP/REG_ENTREPRIS...,NaN,NaN,NaN,NaN,NaN,NaN,osm,attributs/GG/osm,osm_attributes.gpkg,NaN,0.5,0.5,75,75,favorable,point,count,NaN,point,count,NaN,2.0,6.0,zero_if_missing,filtered,1,2056,"parquet, csv, gpkg",TB
8,Commodité,temperature,temperature,True,True,sitg,attributs/GE/temperature,CLIMAT_TEMPERATURE_14H00_P1_2020/CLIMAT_TEMPER...,NaN,NaN,NaN,NaN,NaN,NaN,opendata,attributs/GG/temperature,LST_GrandGeneve_2023.tif,NaN,0.5,0.5,100,250,defavorable,point,raster,temperature,point,raster,temperature,40.0,40.0,keep_nan,filtered,1,2056,"parquet, csv, gpkg",TB
9,Commodité,conflit_md,conflit_md,True,True,sitg,attributs/GE/network_couche_OCT.shp,RP_final_25112025.shp,NaN,NaN,NaN,NaN,NaN,NaN,osm,attributs/GG/osm,osm_attributes.gpkg,NaN,0.5,0.5,1,1,defavorable,line,mean,Partage_us_score,point,sum,severity_score,72.0,2.0,zero_if_missing,filtered,1,2056,"parquet, csv, gpkg",TB
10,Commodité,canopee,canopee,True,True,sitg,attributs/GE/canopee,SIPV_ICA_MNC_2023-SHP/SIPV_ICA_MNC_2023.shp,NaN,NaN,NaN,NaN,NaN,NaN,osm,attributs/GG/osm,osm_attributes.gpkg,NaN,0.5,0.5,30,30,favorable,polygon,area_ratio,NaN,polygon,area_ratio,NaN,NaN,NaN,zero_if_missing,filtered,1,2056,"parquet, csv, gpkg",ok
11,Commodité,banc,banc,True,True,osm,attributs/GE/osm,osm_attributes.gpkg,NaN,NaN,NaN,NaN,NaN,NaN,osm,attributs/GG/osm,osm_attributes.gpkg,NaN,0.5,0.5,10,10,favorable,point,count,NaN,point,presence,NaN,NaN,4.0,zero_if_missing,filtered,1,2056,"parquet, csv, gpkg",ok


In [3]:
segmented_net

,u,v,key,infra_bike,bike_direction,source,highway,cycleway,cycleway:left,cycleway:right,bicycle,oneway,oneway:bicycle,name,maxspeed,len_m,edge_id,length,surface,geometry,segment_id
0,0,1,0,bande_cyclable,both,OSM_highway_bike,residential,None,None,lane,None,None,None,Rue du 31-Décembre,None,33.141014,0,33.141014,None,"LINESTRING (2501153.967 1117899.21, 2501146.00...",000000
1,1,0,0,bande_cyclable,both,OSM_highway_bike,residential,None,None,lane,None,None,None,Rue du 31-Décembre,None,33.141014,0,33.141014,None,"LINESTRING (2501134.7 1117926.175, 2501142.29 ...",000001
2,2,3,0,sur_chaussée,both,OSM_highway_bike,tertiary,None,None,None,None,None,None,Route de la Vie de l'Etraz,50,228.789498,1,50.000000,None,"LINESTRING (2485499.775 1118838.73, 2485502.44...",000002
3,2,3,0,sur_chaussée,both,OSM_highway_bike,tertiary,None,None,None,None,None,None,Route de la Vie de l'Etraz,50,228.789498,1,50.000000,None,"LINESTRING (2485536.487 1118872.654, 2485567.5...",000003
4,2,3,0,sur_chaussée,both,OSM_highway_bike,tertiary,None,None,None,None,None,None,Route de la Vie de l'Etraz,50,228.789498,1,50.000000,None,"LINESTRING (2485572.623 1118907.092, 2485604.0...",000004
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
732159,85510,136495,0,sur_chaussée,both,OSM_highway_bike,service,None,None,None,None,None,None,None,None,29.731631,99157,29.731631,None,"LINESTRING (2490418.359 1105493.044, 2490424.7...",732159
732160,103932,136496,0,sur_chaussée,both,OSM_highway_bike,service,None,None,None,None,None,None,None,None,64.284752,99158,50.000000,None,"LINESTRING (2490517.603 1105416.622, 2490518.3...",732160
732161,103932,136496,0,sur_chaussée,both,OSM_highway_bike,service,None,None,None,None,None,None,None,None,64.284752,99158,14.284752,None,"LINESTRING (2490550.281 1105443.624, 2490556.0...",732161
732162,136496,103932,0,sur_chaussée,both,OSM_highway_bike,service,None,None,None,None,None,None,None,None,64.284752,99158,50.000000,None,"LINESTRING (2490556.961 1105435.402, 2490556.0...",732162


V1 Original

In [4]:
# # Garder uniquement les colonnes 'segment_id', 'vitesse', et 'geometry' dans segmented_net
# segmented_net = segmented_net[['geometry', 'segment_id', 'length']].copy()

# '''# --- ADD THIS BLOCK ---
# # Sample a subset of the network for faster testing
# sample_fraction = 0.7  # 70% of all segments, adjust as needed
# segmented_net = segmented_net.sample(frac=sample_fraction, random_state=42)

# print(f"Testing on a sample of {len(segmented_net)} segments out of the total network.")
# # ----------------------'''

# print('Boucle sur chaque attribut... peut prendre du temps (30 mn)')

# # Boucle sur le derniers attributs, first 3 only for testing
# for _, row in attributs_info.iterrows():
#     if row[include]:
#         attribute_name = row['attribute']
#         method = row['method']
#         how = row['how']
#         value_column = row['value_column']
#         buffer_size = row['buffer_size']
#         geometry_type = row['geometry_type']
#         feature_query_expr = make_feature_query(
#         row.get('filter_column'),
#         row.get('filter_values')
#     )

#         # Charger la couche attribut depuis gpd_attributs
#         print(f"Chargement de la couche: {attribute_name}")
#         attribute_gdf = gpd.read_parquet(f"../../Data/output/step-2/parquet_attributs/{attribute_name}.parquet")
#         attribute_gdf = attribute_gdf.to_crs(segmented_net.crs)

#         # Appliquer la méthode
        
#         if method == "A": # Buffer feature extraction
#             print(f"Applying method {method} for attribute {attribute_name}...")
#             attribute_df = extract_buffer_feature(
#                 segments_gdf = segmented_net,
#                 feature_gdf = attribute_gdf,
#                 feature_name=attribute_name,
#                 geom_kind=geometry_type,
#                 buffer_radius=buffer_size,
#                 how=how,
#                 value_column=value_column,
#                 crs_meter_epsg=operation_crs,
#                 feature_query  = feature_query_expr,
#             )
#             print(f"Buffer feature extracted for {attribute_name} with method {method}")
        
#                 # Debug duplicates
#             if attribute_df[attribute_df.duplicated('segment_id')].shape[0] > 0:
#                 print("\nDEBUG: Found duplicate segment assignments")
#                 dupes = attribute_df[attribute_df.duplicated('segment_id', keep=False)]
#                 print(f"Number of segments with multiple zones: {len(dupes['segment_id'].unique())}")
            
#                 # Keep only the first occurrence for each segment_id
#                 attribute_df = attribute_df.drop_duplicates('segment_id', keep='first')
#                 print("Dropped duplicates, keeping first occurrence")
#             print(f"Spatial join computed for {attribute_name} with method {method}") 
        
#         # Ajoute d'autres méthodes si besoin
        
#         # Ajouter la colonne au GeoDataFrame principal
#         segmented_net[f'{attribute_name}_{method}_{buffer_size}'] = attribute_df[attribute_name].fillna(0)
#         print(f"Attribute {attribute_name} added to segmented_net.")

# # Sauvegarder
# print("Saving step 2 features to parquet...")
# segmented_net.to_crs(target_crs).to_parquet(os.path.join(output_step2_path, "step2_features.parquet"), index=False)


V2

In [5]:
# # Garder uniquement les colonnes 'segment_id', 'vitesse', et 'geometry' dans segmented_net
# segmented_net = segmented_net[['geometry', 'segment_id', 'length']].copy()

# '''# --- ADD THIS BLOCK ---
# # Sample a subset of the network for faster testing
# sample_fraction = 0.7  # 70% of all segments, adjust as needed
# segmented_net = segmented_net.sample(frac=sample_fraction, random_state=42)

# print(f"Testing on a sample of {len(segmented_net)} segments out of the total network.")
# # ----------------------'''

# print('Boucle sur chaque attribut... peut prendre du temps (30 mn)')

# # Boucle sur le derniers attributs, first 3 only for testing
# for idx, row in attributs_info.iterrows():
#     if row[include]:
#         attribute_name = row['attribute']
#         method = row['method']
#         how = row['how']
#         value_column = row['value_column']
#         buffer_size = row['buffer_size']
#         geometry_type = row['geometry_type']
        
#         try:
#             feature_query_expr = make_feature_query(
#                 row.get('filter_column'),
#                 row.get('filter_values')
#             )

#             # Charger la couche attribut depuis gpd_attributs
#             print(f"\n[{idx+1}/{len(attributs_info)}] Chargement de la couche: {attribute_name}")
#             attribute_gdf = gpd.read_parquet(f"../../Data/output/step-2/parquet_attributs/{attribute_name}.parquet")
#             print(f"  → {len(attribute_gdf)} features chargées")
#             attribute_gdf = attribute_gdf.to_crs(segmented_net.crs)

#             # Appliquer la méthode
#             if method == "A": # Buffer feature extraction
#                 print(f"  → Applying method {method} (how={how}, buffer={buffer_size}m)...")
#                 attribute_df = extract_buffer_feature(
#                     segments_gdf = segmented_net,
#                     feature_gdf = attribute_gdf,
#                     feature_name=attribute_name,
#                     geom_kind=geometry_type,
#                     buffer_radius=buffer_size,
#                     how=how,
#                     value_column=value_column,
#                     crs_meter_epsg=operation_crs,
#                     feature_query  = feature_query_expr,
#                 )
#                 print(f"  → Buffer feature extracted")
            
#                 # Debug duplicates
#                 if attribute_df[attribute_df.duplicated('segment_id')].shape[0] > 0:
#                     print("  → WARNING: Found duplicate segment assignments")
#                     dupes = attribute_df[attribute_df.duplicated('segment_id', keep=False)]
#                     print(f"     Segments with duplicates: {len(dupes['segment_id'].unique())}")
                
#                     # Keep only the first occurrence for each segment_id
#                     attribute_df = attribute_df.drop_duplicates('segment_id', keep='first')
#                     print("  → Duplicates dropped (keeping first)")
            
#             # Ajoute d'autres méthodes si besoin
            
#             # Ajouter la colonne au GeoDataFrame principal
#             col_name = f'{attribute_name}_{method}_{buffer_size}'
#             segmented_net[col_name] = attribute_df[attribute_name].fillna(0)
#             print(f"  ✓ Attribute {attribute_name} added to segmented_net as '{col_name}'")
            
#         except Exception as e:
#             print(f"  !!! ERROR processing {attribute_name}: {e}")
#             import traceback
#             traceback.print_exc()
#             # Continue avec l'attribut suivant au lieu de crasher
#             continue

# # Sauvegarder
# print("\n" + "="*60)
# print("Saving step 2 features to parquet...")
# segmented_net.to_crs(target_crs).to_parquet(os.path.join(output_step2_path, "step2_features.parquet"), index=False)
# print(f"✓ Saved {len(segmented_net)} segments with {len(segmented_net.columns)} columns")
# print("="*60)

V3

In [ ]:
import os, time
import numpy as np
import geopandas as gpd
import shapely
from feature_query import make_feature_query  # ta fonction inline

# ========= Réglages de performance =========
BUFFER_RESOLUTION = 2           # précision suffisante
UNION_LINES_FOR_LENGTH_RATIO = True  # éviter le double comptage
ROUND_DECIMALS = 3              # arrondi à 3 décimales pour tous les floats

# ========= Préparer le réseau (CRS + colonnes utiles) =========
if segmented_net.crs is None or segmented_net.crs.to_epsg() != operation_crs:
    segmented_net = segmented_net.to_crs(operation_crs)

segmented_net = segmented_net[['segment_id', 'geometry', 'length']].copy()
segmented_net = segmented_net.drop_duplicates('segment_id')
segmented_net = segmented_net.set_index('segment_id', drop=False)

print('Boucle sur chaque attribut (méthode définie dans la table attributaire)…')


for _, row in attributs_info.iterrows():
    #continue si l'attribut n'est pas inclus dans l'index ("False" ou "Unavailable")
    if not row.get(include, "False"):
        continue

    attribute_name = row['attribute']
    how            = row[how_name]                     # presence / count / mean / sum / area_ratio / length_ratio / raster
    value_column   = row.get(value_column_name, None)
    buffer_size    = float(row[f"buffer_size_{territory}"])
    geometry_type  = str(row[geometry_type_name]).lower().strip()

    print(f"\n→ {attribute_name} | how={how}, geom={geometry_type}, buffer={buffer_size}m")
    t0 = time.perf_counter()

    # 1) Charger la couche attribut + CRS
    attr_path = f"{output_step2_path}/parquet_attributs/{attribute_name}.parquet"
    attribute_gdf = gpd.read_parquet(attr_path)
    if attribute_gdf.crs != segmented_net.crs:
        attribute_gdf = attribute_gdf.to_crs(segmented_net.crs)

    # 2) Filtrer via make_feature_query (inline)
    expr = make_feature_query(row.get('filter_column'), row.get('filter_values'))
    if expr:
        try:
            attribute_gdf = attribute_gdf.query(expr)
        except Exception as e:
            print(f"  ⚠️ Filtre ignoré ({expr}) : {e}")

    # 3) Nettoyage géométries
    attribute_gdf = attribute_gdf[
        attribute_gdf.geometry.notna() & ~attribute_gdf.geometry.is_empty
    ].copy()
    if not attribute_gdf.geometry.is_valid.all():
        attribute_gdf['geometry'] = shapely.make_valid(attribute_gdf.geometry.values)

    print(f"  Taille attribut: {len(attribute_gdf):,} | chargement+filtre: {time.perf_counter()-t0:.1f}s")

    # 5) value_column peut être NaN → None
    if isinstance(value_column, float) and np.isnan(value_column):
        value_column = None

    # 6) Extraction (toujours méthode A)
    t1 = time.perf_counter()
    attribute_df = extract_buffer_feature(
        segments_gdf        = segmented_net.reset_index(drop=True),
        feature_gdf         = attribute_gdf,
        feature_name        = attribute_name,
        geom_kind           = geometry_type,
        how                 = how,
        buffer_radius       = buffer_size,
        value_column        = value_column,
        crs_meter_epsg      = operation_crs,
        predicate           = row.get('predicate', None),
        feature_query       = None,  # déjà filtré
        segment_length_col  = 'length',
        zero_for_missing    = True,
        raster_stats        = row.get('raster_stats', 'mean'),
        buffer_resolution   = BUFFER_RESOLUTION,
    )
    print(f"  ✓ extract_buffer_feature: {time.perf_counter()-t1:.1f}s — {attribute_df.shape[0]} lignes")

    # 7) Sécurité unicité
    if attribute_df.duplicated('segment_id').any():
        ndup = attribute_df[attribute_df.duplicated('segment_id', keep=False)]['segment_id'].nunique()
        print(f"  ⚠️ Duplicates détectés pour {ndup} segments → on garde le premier")
        attribute_df = attribute_df.drop_duplicates('segment_id', keep='first')

    # 8) Assignation + arrondi
    attribute_df = attribute_df.set_index('segment_id')
    colname = f"{attribute_name}_{int(buffer_size)}"  # plus de “_A_” ici

    if colname not in segmented_net.columns:
        segmented_net[colname] = 0.0

    ix = segmented_net.index.intersection(attribute_df.index)
    vals = attribute_df.loc[ix, attribute_name].to_numpy(dtype='float64')
    segmented_net.loc[ix, colname] = np.around(vals, ROUND_DECIMALS)

    print(f"  → Colonne ajoutée: {colname} (arrondie à {ROUND_DECIMALS} décimales)")

# 9) Sauvegarde (et arrondi global final)
print("\nSaving step 2 features to parquet…")
if segmented_net.crs != target_crs:
    segmented_net = segmented_net.to_crs(target_crs)

float_cols = segmented_net.select_dtypes(include=['float', 'float32', 'float64']).columns
segmented_net[float_cols] = segmented_net[float_cols].astype('float64').round(ROUND_DECIMALS)

out_path = os.path.join(output_step2_path, "step2_features.parquet")
segmented_net.to_parquet(out_path, index=False)
print(f"✔️ Enregistré: {out_path}")

Boucle sur chaque attribut (méthode définie dans la table attributaire)…

→ lac_cours_deau | how=presence, geom=polygon, buffer=50.0m
  Taille attribut: 8,224 | chargement+filtre: 0.1s


In [ ]:

segmented_net.head(20)


In [ ]:
#import step2_features and convert to gpkg for QGIS use
segmented_net = gpd.read_parquet(os.path.join(output_step2_path, "step2_features.parquet"))
segmented_net.to_file(os.path.join(output_step2_path, "step2_features.gpkg"), driver="GPKG")
